# Tier Comparison — V1 vs V2 vs V3 on the same alerts

This is the headline result of the project: **does climbing the autonomy ladder actually pay off?**

Every tier runs against the same V2 test set (now 20 alerts including 5 deliberate refusal cases — alerts that don't map to any internal runbook). We measure on three axes:

1. **Quality** — routing accuracy, retrieval recall, refusal accuracy, LLM-as-judge plan scores.
2. **Effort** — number of LLM calls / tool calls per alert.
3. **Latency** — wall-clock seconds per alert.

The expected story:
- V1 (router) wins on latency / cost — it's one call.
- V2 (planner) wins on plan quality for in-distribution alerts.
- V3 (agent) is the only tier that visibly *investigates* edge cases; it should dominate on refusal cases (the agent should refuse to invent a runbook) and on alerts where the right runbook isn't obvious from BM25 alone.

In [ ]:
import os, sys, time, json
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')
assert os.environ.get('OPENAI_API_KEY'), 'set OPENAI_API_KEY in ../.env'

import pandas as pd
from src import RouterAgent, PlannerAgent, ReactAgent, RunbookIndex
from src import evaluate_router, evaluate_planner

tests = pd.read_csv('../data/test_cases_v2.csv')
print(f'{len(tests)} test alerts ({tests["expected_runbook"].isna().sum() + (tests["expected_runbook"] == "").sum()} refusal)')
tests.head(3)

## 1. Run all three tiers on the same set

In [ ]:
router = RouterAgent(prompt_version='improved')
index = RunbookIndex(runbook_dir='../data/runbooks')
planner = PlannerAgent(router=router, index=index, prompt_version='improved')
agent = ReactAgent(index=index, max_iterations=6)

# --- V1: router-only ---
v1_only = tests[['id', 'alert_text', 'expected_category']]
t0 = time.monotonic()
v1_result = evaluate_router(router, v1_only, progress=False)
v1_elapsed = time.monotonic() - t0
print(f'V1 done in {v1_elapsed:.1f}s')

In [ ]:
# --- V2: full planner pipeline (judge enabled) ---
t0 = time.monotonic()
v2_result = evaluate_planner(planner, tests, judge=True, progress=False)
v2_elapsed = time.monotonic() - t0
print(f'V2 done in {v2_elapsed:.1f}s')

In [ ]:
# --- V3: ReAct agent. Track per-alert tool-call count + latency for the comparison table ---
v3_rows = []
t_total = time.monotonic()
for row in tests.itertuples(index=False):
    t0 = time.monotonic()
    run = agent.run(row.alert_text)
    elapsed = time.monotonic() - t0
    expected_rb = (str(row.expected_runbook) or '').strip().lower()
    is_refusal = expected_rb in {'', 'nan', 'none'}
    category = ReactAgent.get_category(run).value
    primary_rb = (run.plan or {}).get('primary_runbook')
    v3_rows.append({
        'id': row.id,
        'expected_category': row.expected_category,
        'predicted_category': category,
        'category_correct': category == row.expected_category,
        'is_refusal': is_refusal,
        'refusal_correct': (primary_rb in (None, 'null')) if is_refusal else None,
        'expected_runbook': '' if is_refusal else row.expected_runbook,
        'primary_runbook': primary_rb,
        'in_top_k': True if (not is_refusal and any(
            tc.name == 'search_runbooks' and any(
                h.get('runbook_id') == row.expected_runbook for h in (tc.result or [])
            ) for tc in run.tool_calls
        )) else (None if is_refusal else False),
        'tool_calls': len(run.tool_calls),
        'iterations': run.num_iterations,
        'terminated_via': run.terminated_via,
        'elapsed_s': elapsed,
    })
v3 = pd.DataFrame(v3_rows)
v3_elapsed = time.monotonic() - t_total
print(f'V3 done in {v3_elapsed:.1f}s')
v3.head()

## 2. Headline comparison table

One table. This is what a reviewer should see first.

In [ ]:
def _mean(s):
    s = s.dropna()
    return float(s.mean()) if len(s) else None

comparison = pd.DataFrame([
    {
        'tier': 'V1 router',
        'routing_accuracy': v1_result.accuracy,
        'recall@1': None,
        'recall@k': None,
        'refusal_accuracy': None,           # router has no notion of refusal
        'avg_groundedness': None,
        'avg_LLM_or_tool_calls': 1.0,
        'wall_clock_total_s': round(v1_elapsed, 1),
        'per_alert_s': round(v1_elapsed / max(1, len(tests)), 2),
    },
    {
        'tier': 'V2 planner',
        'routing_accuracy': v2_result.routing_accuracy,
        'recall@1': v2_result.recall_at_1,
        'recall@k': v2_result.recall_at_k,
        'refusal_accuracy': v2_result.refusal_accuracy,
        'avg_groundedness': v2_result.avg_groundedness,
        'avg_LLM_or_tool_calls': 2.0,        # router call + planner call
        'wall_clock_total_s': round(v2_elapsed, 1),
        'per_alert_s': round(v2_elapsed / max(1, len(tests)), 2),
    },
    {
        'tier': 'V3 ReAct agent',
        'routing_accuracy': _mean(v3['category_correct'].astype(float)),
        'recall@1': None,                    # V3 doesn't have a strict top-1 concept
        'recall@k': _mean(v3[~v3['is_refusal']]['in_top_k'].astype(float)),
        'refusal_accuracy': _mean(v3[v3['is_refusal']]['refusal_correct'].astype(float)),
        'avg_groundedness': None,            # plan-grounding judge call done in V2 only here; could be added
        'avg_LLM_or_tool_calls': _mean(v3['tool_calls'].astype(float)),
        'wall_clock_total_s': round(v3_elapsed, 1),
        'per_alert_s': round(v3_elapsed / max(1, len(tests)), 2),
    },
])
comparison

## 3. Where each tier wins or loses

The interesting cases — where V3 makes a different decision from V2 — are the part to inspect by eye.

In [ ]:
v2_per_alert = v2_result.predictions.set_index('id')
v3_per_alert = v3.set_index('id')
joined = v2_per_alert[['expected_category', 'predicted_category', 'is_refusal', 'refusal_correct', 'in_top_k', 'primary_runbook']].join(
    v3_per_alert[['predicted_category', 'refusal_correct', 'in_top_k', 'primary_runbook', 'tool_calls']],
    lsuffix='_v2', rsuffix='_v3',
)
disagreements = joined[
    (joined['predicted_category_v2'] != joined['predicted_category_v3']) |
    (joined['refusal_correct_v2'] != joined['refusal_correct_v3']) |
    (joined['in_top_k_v2'] != joined['in_top_k_v3'])
]
print(f'{len(disagreements)} alerts where V2 and V3 disagree')
disagreements

In [ ]:
# Refusal slice — only the alerts that have no matching runbook
refusal_only = joined[joined['is_refusal']][['refusal_correct_v2', 'refusal_correct_v3', 'primary_runbook_v2', 'primary_runbook_v3']]
refusal_only

## Takeaways

Patterns to look for in the table above:

- **V1's routing accuracy is high but it can't refuse.** A router that returns a category for every input is doing harm on the refusal cases — there's no "none of these" option.
- **V2 has the same routing as V1 by construction, but adds retrieval.** Recall@k on in-distribution alerts should be high; refusal accuracy depends entirely on whether the prompt makes the planner willing to set `primary_runbook=null`.
- **V3 spends 3-6x more tool calls per alert** but is the only tier whose investigation depends on what it sees. On refusal cases it should dominate — the agent that calls `search_runbooks` and sees zero relevant hits is the only one with the information to refuse honestly.

If the numbers in your run don't match this story, that's the most interesting thing in the project. Failure to climb the ladder cleanly is a signal that V2's pipeline is over-fitted to the test set, or that V3's prompt isn't strict enough about refusing. Both are real things to fix.